# Fitting a steady point-source with the public 14-year IceCube track data

This tutorial shows how to fit a neutrino point-like steady source (time-integrated) using the IceCube public 14-year point-source data with SkyLLH.

We assume that you have installed the SkyLLH package in your environment. If that's not the case you'll need to add the path to the SkyLLH folder to your PYTHONPATH.

In [2]:
import numpy as np

## Initializing a configuration dictionary

First we have to create a config instance.

In [3]:
from skyllh.core.config import Config

cfg = Config()

In [4]:
cfg

{'multiproc': {'ncpu': None},
 'logging': {'log_level': 'INFO',
  'log_format': '%(asctime)s %(processName)s %(name)s %(levelname)s: %(message)s',
  'enable_tracing': False},
 'project': {'working_directory': '.'},
 'repository': {'base_path': None, 'download_from_origin': True},
 'units': {'internal': {'angle': Unit("rad"),
   'energy': Unit("GeV"),
   'length': Unit("cm"),
   'time': Unit("s")},
  'defaults': {'fluxes': {'angle': Unit("rad"),
    'energy': Unit("GeV"),
    'length': Unit("cm"),
    'time': Unit("s")}}},
 'datafields': {'run': 4,
  'ra': 4,
  'dec': 4,
  'ang_err': 4,
  'time': 4,
  'log_energy': 4,
  'true_ra': 8,
  'true_dec': 8,
  'true_energy': 8,
  'mcweight': 8},
 'caching': {'pdf': {'MultiDimGridPDF': False}}}

## Getting the datasets

The dataset path can be set in the config. If not set, the notebook downloads it directly from the IceCube Dataverse (origin).


In [5]:
# TODO: Test that downloading works when we have the final release on the dataverse!
# For the moment I set the path to my own local path.
cfg['repository']['base_path'] = '/Users/chiarabellenghi/Work/data_release/'

To create the correct dataset object, we need to import the 14-yr dataset definition:

In [6]:
from skyllh.datasets.i3.PublicData_14y_ps import create_dataset_collection

The collection of datasets can be created using the ``create_dataset_collection`` function.

It takes the repository `base_path` and `sub_path_fmt` as optional arguments. They are internally comabined into the folder path `<base_path>/<sub_path_fmt>` and the events, irfs, and uptime sub-folders are expected to be found in there. 

We have defined the base path in the configuration file, so we can leave that one as None. The `sub_path_fmt` for my dataset is 'PublicData_14y'

In [7]:
create_dataset_collection?

dsc = create_dataset_collection(cfg=cfg, sub_path_fmt='PublicData_14y')

Signature: create_dataset_collection(cfg, base_path=None, sub_path_fmt=None)
Docstring:
# Defines the dataset collection for IceCube's 14-year
# point-source public data, which is available at
# TBD

Parameters
----------
cfg : instance of Config
    The instance of Config holding the local configuration.
base_path : str | None
    The base path of the data files. The actual path of a data file is
    assumed to be of the structure <base_path>/<sub_path>/<file_name>.
    If None, use the default path ``cfg['repository']['base_path']``.
sub_path_fmt : str | None
    The sub path format of the data files of the public data sample.
    If None, use the default sub path format
    'icecube_10year_ps'.

Returns
-------
dsc : DatasetCollection
    The dataset collection containing all the seasons as individual
    I3Dataset objects.
File:      ~/Work/skyllh_dev/skyllh/skyllh/datasets/i3/PublicData_14y_ps.py
Type:      function

``dataset_names`` property provides a list of all the data sets defined in the data set collection of the public point-source data.   

The individual data sets ``IC86_I``, ``IC86_II``, ``IC86_III``, ``IC86_IV``, ``IC86_V``, ``IC86_VI``, ``IC86_VII``, ``IC86_VIII``, ``IC86_IX``, ``IC86_X``, and ``IC86_XI`` are also available as a single combined data set ``IC86_I-IX``, because these data sets share the same detector simulation and event selection. Hence, we can get a list of data sets via the access operator ``[dataset1, dataset2, ...]`` of the ``dsc`` instance:

In [8]:
print(dsc.dataset_names)
datasets = dsc['IC40', 'IC59', 'IC79', 'IC86_I-XI']

['IC40', 'IC59', 'IC79', 'IC86_I', 'IC86_I-XI', 'IC86_II', 'IC86_III', 'IC86_IV', 'IC86_IX', 'IC86_V', 'IC86_VI', 'IC86_VII', 'IC86_VIII', 'IC86_X', 'IC86_XI']


## Getting the analysis

The analysis for doing a point-source search as in, e.g., https://doi.org/10.1103/PhysRevLett.124.051103 is pre-defined. This analysis instance can be created via the `create_analysis` method of the `time_integrated_ps` module of the public data interface.

In [9]:
from skyllh.analyses.i3.publicdata_ps.time_integrated_ps import create_analysis

help(create_analysis)

Help on function create_analysis in module skyllh.analyses.i3.publicdata_ps.time_integrated_ps:

create_analysis(
    cfg,
    datasets,
    source,
    refplflux_Phi0=1,
    refplflux_E0=1000.0,
    refplflux_gamma=2.0,
    refplflux_Ec=inf,
    ns_seed=10.0,
    ns_min=0.0,
    ns_max=1000.0,
    gamma_seed=3.0,
    gamma_min=1.0,
    gamma_max=4.0,
    kde_smoothing=False,
    minimizer_impl='LBFGS',
    minimizer_max_rep=100,
    compress_data=False,
    keep_data_fields=None,
    evt_sel_delta_angle_deg=10,
    construct_sig_generator=True,
    tl=None,
    ppbar=None,
    logger_name=None
)
    Creates the Analysis instance for this particular analysis.

    Parameters
    ----------
    cfg : instance of Config
        The instance of Config holding the local configuration.
    datasets : list of Dataset instances
        The list of Dataset instances, which should be used in the
        analysis.
    source : PointLikeSource instance
        The PointLikeSource instance definin

Defining the Source: Here we use NGC 1068 as the source

Once the source is defined the ``create_analysis`` function can create the analysis instance. By default, it creates the source instance emitting a powerlaw flux with spectral index 2.0

In [10]:
from skyllh.core.source_model import PointLikeSource

src_ra = 40.67  # degrees
src_dec = -0.01  # degrees

source = PointLikeSource(ra=np.radians(src_ra), dec=np.radians(src_dec))

ana = create_analysis(cfg=cfg, datasets=datasets, source=source)

  0%|          | 0/4 [00:00<?, ?it/s]

100%|██████████| 136/136 [00:00<00:00, 11974.17it/s]


## Initializing a trial

After the `Analysis` instance was created, the point-source analysis can be run. To do so the analysis needs to be initialized with some _trial data_.
For instance we could initialize the analysis with the experimental data to "unblind" the analysis afterwards. 

Technically the `TrialDataManager` of each log-likelihood ratio function, i.e. dataset, is initialized with data.

The `Analysis` object provides the method `initialize_trial` to initialize a trial with data. It takes a list of `DataFieldRecordArray` instances holding the events. If we want to initialize a trial with the experimental data, we can get that list from the `Analysis` instance itself:

In [11]:
events_list = [data.exp for data in ana.data_list]
ana.initialize_trial(events_list)

## Maximizing the log-likelihood ratio function

After initializing a trial, we can maximize the LLH ratio function using the `maximize_llhratio` method of the `Analysis` class. This method requires a ``RandomStateService`` instance in case the minimizer does not succeed and a new set of initial values for the fit parameters need to get generated. The method returns a 4-element tuple. The first element is the set of fit parameters used in the maximization. The second element is the value of the LLH ration function at its maximum. The third element is the array of the fit parameter values at the maximum, and the forth element is the status dictionary of the minimizer.

In [10]:
from skyllh.core.random import RandomStateService

rss = RandomStateService(seed=1)

In [11]:
(log_lambda_max, fitparam_values, status) = ana.llhratio.maximize(rss)

In [12]:
print(f'log_lambda_max = {log_lambda_max}')
print(f'fitparam_values = {fitparam_values}')
print(f'status = {status}')

log_lambda_max = 14.576758408087755
fitparam_values = [80.09381868  3.20884803]
status = {'grad': array([-3.62369895e-06, -2.31360821e-04]), 'task': 'CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH', 'funcalls': 13, 'nit': 10, 'warnflag': 0, 'skyllh_minimizer_n_reps': 0, 'n_llhratio_func_calls': 13}


## Calculating the test-statistic

Using the maximum of the LLH ratio function and the fit parameter values at the maximum we can calculate the test-statistic using the `calculate_test_statistic` method of the `Analysis` class:

In [13]:
TS = ana.calculate_test_statistic(log_lambda_max, fitparam_values)
print(f'TS = {TS:.3f}')

TS = 29.154


## Unblinding the data

Above, we perform the analysis following pedagogical steps: we initialize the analysis with a trial of the experimental data, maximize the log-likelihood ratio function for all given experimental data events, and calculate the test-statistic value.

However, the same calculations can be performed all togethe using the ``unblind`` method of the analysis instance, which returns the test statistic and the best fit parameters directly. This method requires a ``RandomStateService`` instance in case the minimizer does not succeed and a new set of initial values for the fit parameters need to get generated.

Note that the ``unblind`` method initializes the trials on the experimental data only. The trials are initialised internally, and ``initialize_trial`` then need not be explicitly run.

The methods to generate and analyze pseudo-data are illustrated in the `generating_pseudo_experiments.ipynb` notebook.

In [17]:
help(ana.unblind)

Help on method unblind in module skyllh.core.analysis:

unblind(minimizer_rss, tl=None) method of skyllh.core.analysis.SingleSourceMultiDatasetLLHRatioAnalysis instance
    Evaluates the unscrambled data, i.e. unblinds the data.

    Parameters
    ----------
    minimizer_rss : instance of RandomStateService
        The instance of RandomStateService that should be used by the
        minimizer to generate new random initial fit parameter values.
    tl : instance of TimeLord | None
        The optional instance of TimeLord that should be used to time the
        maximization of the LLH ratio function.

    Returns
    -------
    TS : float
        The test-statistic value.
    global_params_dict : dict
        The dictionary holding the global parameter names and their
        best fit values. It includes fixed and floating parameters.
    status : dict
        The status dictionary with information about the performed
        minimization process of the negative of the log-likeliho

In [18]:
from skyllh.core.random import RandomStateService

rss = RandomStateService(seed=1)

(ts, x, status) = ana.unblind(rss)

In [19]:
print(f'TS = {ts:.3f}')
print(f'ns = {x["ns"]:.2f}')
print(f'gamma = {x["gamma"]:.2f}')
status

TS = 29.154
ns = 80.09
gamma = 3.21


{'grad': array([-3.62369979e-06, -2.31360814e-04]),
 'task': 'CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH',
 'funcalls': 13,
 'nit': 10,
 'warnflag': 0,
 'skyllh_minimizer_n_reps': 0,
 'n_llhratio_func_calls': 13}

## Calculating the corresponding flux normalization 

By default the analysis is created with a flux normalization of 1 $\text{GeV}^{-1}\text{s}^{-1}\text{cm}^{-2}\text{sr}^{-1}$ at 1 TeV (see `refplflux_Phi0` and `refplflux_E0` arguments of the `create_analysis` method). The analysis instance has the method `calculate_fluxmodel_scaling_factor` that calculates the scaling factor the reference flux normalization has to be multiplied with to represent a given analysis result, i.e. $n_{\text{s}}$ and $\gamma$ value. This function takes the detected mean $n_{\text{s}}$ value as first argument and the list of source parameter values as second argument:

In [17]:
scaling_factor = ana.calculate_fluxmodel_scaling_factor(x['ns'], [x['ns'], x['gamma']])
print(f'Flux scaling factor = {scaling_factor:.3e}')

Flux scaling factor = 2.869e-14


Hence, our result corresponds to a power-law flux of:

In [ ]:
print(f'{scaling_factor:.3e} (E/1000 GeV)^{{ -{x["gamma"]:.2f} }} 1/(GeV s cm^2 sr)')

2.869e-14 (E/1000 GeV)^{-3.21} 1/(GeV s cm^2 sr)


Illustrating the difference of fits between the 10-year and the 14-year dataset. To compute the test statistic and the best fit parameters for the 10-year dataset, the same workflow as used for the 14-year dataset.  

In [22]:
from skyllh.core.config import Config

cfg_10 = Config()

In [23]:
from skyllh.datasets.i3.PublicData_10y_ps import create_dataset_collection

In [24]:
dsc = create_dataset_collection(cfg=cfg_10, base_path='/Users/amithan/Downloads/icecube_10year_ps')

datasets = dsc['IC40', 'IC59', 'IC79', 'IC86_I', 'IC86_II-VII']

In [25]:
from skyllh.analyses.i3.publicdata_ps.time_integrated_ps import create_analysis
from skyllh.core.source_model import PointLikeSource

src_ra = 40.67  # degrees
src_dec = -0.01  # degrees

source = PointLikeSource(ra=np.radians(src_ra), dec=np.radians(src_dec))
ana = create_analysis(cfg=cfg_10, datasets=datasets, source=source)

events_list = [data.exp for data in ana.data_list]

100%|██████████| 170/170 [00:00<00:00, 509.21it/s]


In [ ]:
from skyllh.core.random import RandomStateService

rss = RandomStateService(seed=1)

(ts, x, status) = ana.unblind(rss)
print(f'TS = {ts:.3f}')
print(f'ns = {x["ns"]:.2f}')
print(f'gamma = {x["gamma"]:.2f}')
## Calculating the corresponding flux normalization
scaling_factor = ana.calculate_fluxmodel_scaling_factor(x['ns'], [x['ns'], x['gamma']])
print(f'Flux scaling factor = {scaling_factor:.3e}')

print(f'{scaling_factor:.3e} (E/1000 GeV)^{{ -{x["gamma"]:.2f} }} 1/(GeV s cm^2 sr)')

TS = 19.379
ns = 54.75
gamma = 3.16
Flux scaling factor = 3.017e-14
3.017e-14 (E/1000 GeV)^{-3.16} 1/(GeV s cm^2 sr)
